# Code Attribution

Portions of this code are adapted from:

**FLASH: A Comprehensive Approach to Intrusion Detection via Provenance Graph Representation Learning**

Source: [https://github.com/DART-Laboratory/Flash-IDS](https://github.com/DART-Laboratory/Flash-IDS)


**threaTrace: Detecting and Tracing Host-based Threats in Node Level Through Provenance Graph Learning**

Source: [https://github.com/threaTrace-detector/threaTrace](https://github.com/threaTrace-detector/threaTrace)

## Importing required packages

In [ ]:
import os
from tqdm import tqdm
import torch
import torch.nn.functional as F
import pandas as pd
from collections import defaultdict
import numpy as np
import re
import gdown
import requests
import pickle
from torch_geometric.nn import SAGEConv
from sklearn.metrics.pairwise import cosine_similarity
from utils_cadets.data_process_train import *
from utils_cadets.data_process_test import *
from utils_cadets.evaluate_darpatc import *

## Model Class definition

In [ ]:
class SAGENet(torch.nn.Module):
	def __init__(self, in_channels, out_channels):
		super(SAGENet, self).__init__()	
		self.conv1 = SAGEConv(in_channels, 32, normalize=False)
		self.conv2 = SAGEConv(32, out_channels, normalize=False)

	def forward(self, x, edge_index):
		x = self.conv1(x, edge_index)
		x = F.relu(x)
		x = F.dropout(x, training=self.training)
		x = self.conv2(x, edge_index)
		return F.log_softmax(x, dim=1)

## Download Dataset and alarm/Ground Truth files

In [ ]:
FOLDER_URL = "https://drive.google.com/drive/folders/1oFm7XTEB1cmtHHHmBWSjC5WRX29vjs2b?usp=sharing"

output_dir = os.path.abspath("../groundtruth/cadets")
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=FOLDER_URL,
    output=output_dir,
    quiet=False,
    use_cookies=True
)

In [ ]:
os.makedirs("cadets_resources", exist_ok=True)

file_ids = [
    "1Df-IkZMQHq-AVqF8D8Dj6sGdwREH0oCm",
    "15ADm9IeW-HyRlcHK3S4aEZ5TFdFy1brx",
    "1g7Fh_FnRRXl-FH-cxAQMFC5pqQFTtjsB",
    "1v0n3GKsLQiTmXDO3MWbHRTH7N61U6t4_"
]

def get_filename(file_id):
    url = f"https://drive.google.com/file/d/{file_id}/view"
    html = requests.get(url).text
    match = re.search(r'<meta property="og:title" content="(.+?)"', html)
    return match.group(1) if match else file_id

for fid in file_ids:
    filename = get_filename(fid)
    out_path = os.path.join("cadets_resources", filename)
    url = f"https://drive.google.com/uc?id={fid}"
    
    print(f"⬇️ Downloading: {filename}")
    gdown.download(url, out_path, quiet=False)

## Download the models and features

In [ ]:
FOLDER_URL = "https://drive.google.com/drive/folders/1RXKLjPgYMO0xqG14SlqXmSTyCkHJCaPR"

output_dir = os.path.abspath("../models/cadets")
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=FOLDER_URL,
    output=output_dir,
    quiet=False,
    use_cookies=False
)

print(f"All files downloaded into: {output_dir}")

In [ ]:
FOLDER_URL = "https://drive.google.com/drive/folders/17Bl_-glOE-DZTXshPQs_FCmy2_LbVYwf"

output_dir = os.path.abspath("../models/cadets")
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=FOLDER_URL,
    output=output_dir,
    quiet=False,
    use_cookies=False
)

print(f"All files downloaded into: {output_dir}")

## Load test data

In [ ]:
with open("cadets_resources/cadets_dataframe.pkl", "rb") as file:
    df, NODES, phrases, labels, edges_index, mapp, edges_ids, edges_type, index_map, GT_mal = pickle.load(file)

# --- Load test dataset ---
path = 'cadets_resources/cadets_test.txt' 
data1, feature_num, label_num, adj, adj2, nodeA = MyDatasetA(path, 'cadets')
dataset_test = TestDatasetA(data1)
data_test = dataset_test[0] 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
label_num

## Function that evaluates the model and returns metrics

In [ ]:
def test_one_model(mask, model, device, data, thre, nodeA):
    model.eval()
    data = data.to(device)

    # start with all nodes flagged as suspicious
    updated_mask = torch.ones(data.num_nodes, dtype=torch.bool, device=device)


    # Convert malicious list to set for O(1) lookup
    malicious_set = set(nodeA)

    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        pro = torch.softmax(out, dim=1)
        pro1 = pro.max(1)

        # suppress top prediction to find 2nd-best for thresholding
        pro_clone = pro.clone()
        pro_clone[torch.arange(len(pro)), pro1[1]] = -1
        pro2 = pro_clone.max(1)

        # apply threshold logic
        uncertain_mask = (pro1[0] / pro2[0]) < thre
        pred[uncertain_mask] = 100  # dummy label for uncertain predictions

        # evaluation
        TP, FP, TN, FN = 0, 0, 0, 0
        test_nodes = mask.nonzero(as_tuple=False).view(-1).tolist()

        for i in tqdm(test_nodes, desc="Evaluating nodes"):
            flagged = (pred[i] != data.y[i])  # mismatch = flagged
            
            if not flagged:
                updated_mask[i] = False   # node is cleared

            if i in malicious_set:  # malicious node
                if flagged:
                    TP += 1
                else:
                    FN += 1
            else:  # benign node
                if flagged:
                    FP += 1
                else:
                    TN += 1

        # metrics
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0

        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()

    return acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask


## Evaluating before evasion

In [ ]:
thre = 1.5

model_good = SAGENet(feature_num, label_num).to(device)

print(feature_num, label_num)

load_path = "../models/cadets/cadets_model_weights.pth"
model_good.load_state_dict(torch.load(load_path, map_location=device))

acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask = test_one_model(data_test.test_mask, model_good, device, data_test, thre, nodeA)

print(f"Test Accuracy: {acc:.4f}")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, FPR: {fpr:.4f}")

## Functions to prepare the Graph

In [ ]:
def add_node_properties(nodes, node_id, properties):
    if node_id not in nodes:
        nodes[node_id] = []
    nodes[node_id].extend(properties)

def update_edge_index(edges, edge_index, index):
    for src_id, dst_id in edges:
        src = index[src_id]
        dst = index[dst_id]
        edge_index[0].append(src)
        edge_index[1].append(dst)

def prepare_graph(df):
    nodes, labels, edges = {}, {}, []
    dummies = {'SUBJECT_PROCESS': 0, 'FILE_OBJECT_FILE': 1, 'FILE_OBJECT_UNIX_SOCKET': 2, 
               'UnnamedPipeObject': 3, 'NetFlowObject': 4, 'FILE_OBJECT_DIR': 5}
    
    for _, row in df.iterrows():
        action = row["action"]
        properties = [row['exec'], action] + ([row['path']] if row['path'] else [])
        
        actor_id = row["actorID"]
        add_node_properties(nodes, actor_id, properties)
        labels[actor_id] = dummies[row['actor_type']]

        object_id = row["objectID"]
        add_node_properties(nodes, object_id, properties)
        labels[object_id] = dummies[row['object']]

        edges.append((actor_id, object_id))

    features, feat_labels, edge_index, index_map = [], [], [[], []], {}
    for node_id, props in nodes.items():
        features.append(props)
        feat_labels.append(labels[node_id])
        index_map[node_id] = len(features) - 1

    update_edge_index(edges, edge_index, index_map)

    return features, feat_labels, edge_index, list(index_map.keys())

## Evasion

In [ ]:
with open("cadets_resources/cadets_dataframe.pkl", "rb") as file:
    df, NODES, phrases, labels, edges_index, mapp, edges_ids, edges_type, index_map, GT_mal = pickle.load(file)

# --- Load test dataset ---
path = 'cadets_resources/cadets_test.txt' 
data1, feature_num, label_num, adj, adj2, nodeA = MyDatasetA(path, 'cadets')
dataset_test = TestDatasetA(data1)
data_test = dataset_test[0] 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# Find out the indices of malicious nodes using mapping

In [ ]:

indices_of_malicious_nodes = []
for id in GT_mal:
    if id in mapp:
        indices_of_malicious_nodes.append(mapp.index(id))

## Step 1: Benign Node Selection

In [ ]:
def split_malicious_nodes_by_label(malicious_indices, labels):

    r"""Groups malicious nodes based on their labels.

    :param malicious_indices: List of indices for malicious nodes.
    :param labels: List of labels for all nodes.
    :return: Dictionary with label as key and list of malicious node indices as values.
    """

    malicious_groups = defaultdict(list)

    for node_idx in malicious_indices:
        node_label = labels[node_idx]
        malicious_groups[node_label].append(node_idx)

    return dict(malicious_groups)

malicious_groups = split_malicious_nodes_by_label(indices_of_malicious_nodes, labels)

print("The number of malicious nodes for each label is as follows:")

for label, nodes in malicious_groups.items():
    print(f"Label {label}: {len(nodes)}")  

In [ ]:
def split_nodes_by_label(nodes, labels):

    r"""Groups all nodes based on their labels.

    :param nodes: List of phrases (interactions) for all nodes.
    :param labels: List of labels for all nodes.
    :return: Dictionary with label as key and list of node indices as values.
    """

    node_groups = defaultdict(list)
    for node_idx, _ in enumerate(nodes):
        node_label = int(labels[node_idx].item()) 
        node_groups[node_label].append(node_idx)
    return dict(node_groups)

all_nodes_grouped = split_nodes_by_label(data_test.x, data_test.y)

print("The number of nodes (either malicious or benign) for each label is as follows:")
for label, nodes in sorted(all_nodes_grouped.items()):
    print(f"Label {label}: {len(nodes)}")


In [ ]:
def split_nodes_by_label_exclude_malicious(nodes, labels, indices_of_malicious_nodes):

    r"""Groups benign (non-malicious) nodes based on their labels.

    :param nodes: List of phrases (interactions) for all nodes.
    :param labels: List of labels for all nodes.
    :param indices_of_malicious_nodes: List of indices for malicious nodes.
    :return: Dictionary with label as key and list of benign node indices as values.
    """

    node_groups = defaultdict(list)
    malicious_set = set(indices_of_malicious_nodes)  

    for node_idx, _ in enumerate(nodes):
        if node_idx not in malicious_set:
            node_label = int(labels[node_idx].item()) 
            node_groups[node_label].append(node_idx)

    return dict(node_groups)


all_nodes_grouped_excluding_malicious = split_nodes_by_label_exclude_malicious(
    data_test.x, data_test.y, indices_of_malicious_nodes
)

print("The number of candidate benign nodes for each label is as follows:")
for label, nodes in sorted(all_nodes_grouped_excluding_malicious.items()):
    print(f"Label {label}: {len(nodes)}")


## Step 2: Minimal Interaction Selection

In [ ]:
def filter_nodes_by_phrase_length(node_groups, phrases, min_len=50, max_len=150): 

    r"""Filters the benign nodes in each label based on the number of interactions (phrase length).

    :param node_groups: Dictionary with label as key and list of benign node indices as values.
    :param phrases: List of phrases (interactions) for all nodes.
    :param min_len: Minimum number of interactions (hyperparameter).
    :param max_len: Maximum number of interactions (hyperparameter).
    :return: Dictionary with label as key and list of filtered benign node indices as values.
    """

    filtered_groups = {}

    for label, node_indices in node_groups.items():
        filtered_groups[label] = [
            idx for idx in node_indices if min_len <= len(phrases[idx]) <= max_len
        ]

    return filtered_groups


filtered_nodes_by_interactions = filter_nodes_by_phrase_length(all_nodes_grouped_excluding_malicious, phrases)

print("The number of candidate benign nodes for each label is as follows:")

for label, nodes in filtered_nodes_by_interactions.items():
    print(f"Label {label}: {len(nodes)}")  

## Step 3: Contextual Consistency Filtering

In [ ]:
def compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes):

    r"""Orders the benign nodes within each label based on their contextual similarity to the malicious nodes in the same label in descending order.

    :param malicious_groups: Dictionary with label as key and list of malicious node indices as values.
    :param filtered_nodes_grouped: Dictionary with label as key and list of filtered benign node indices as values.
    :param nodes: Feature vectors (embeddings) for all nodes.
    :return: Dictionary with label as key and list of sorted benign node indices as values.
    """

    top_nodes_by_similarity = {}

    for label in tqdm(malicious_groups.keys()):
        if label in filtered_nodes_grouped and len(filtered_nodes_grouped[label]) > 0:
            malicious_indices = malicious_groups[label]
            filtered_indices = filtered_nodes_grouped[label]

            # Get feature vectors for malicious and filtered nodes
            malicious_vectors = np.array([nodes[idx] for idx in malicious_indices])
            filtered_vectors = np.array([nodes[idx] for idx in filtered_indices])

            # Compute cosine similarity (malicious nodes vs filtered nodes)
            similarities = cosine_similarity(filtered_vectors, malicious_vectors)  # Shape: (filtered, malicious)

            # Take the max similarity score for each filtered node
            max_similarities = similarities.max(axis=1)  # Max similarity per filtered node

            # Use np.argsort to get the indices that would sort the max_similarities array
            sorted_indices = np.array(filtered_indices)[np.argsort(-max_similarities)]  # Negative for descending order

            # Select top most similar nodes for the current label
            top_nodes_by_similarity[label] = sorted_indices

    return top_nodes_by_similarity

top_nodes_by_similarity = compute_top_similar_nodes(malicious_groups, filtered_nodes_by_interactions, data_test.x.cpu().numpy())

print("The number of candidate benign nodes for each label is as follows:")

for label, top_nodes in top_nodes_by_similarity.items():
    print(f"Label {label}: {len(top_nodes)}") 

## Graph Structure Adjustment

In [ ]:
def graph_modification(df, indices_of_malicious_nodes, best_benign_replacements, mapp, labels):

    r"""
    Modifies the graph by adding edges between malicious and selected benign nodes.
    :param df: Original DataFrame containing graph edges.
    :param indices_of_malicious_nodes: List of indices for malicious nodes.
    :param best_benign_replacements: Dictionary with label as key and the most confident benign node index as value.
    :param mapp: List mapping node indices to node IDs.
    :param labels: List of labels for all nodes.
    :return: Modified DataFrame with new edges added.
    """
    # Prepare the DataFrame for modification
    print("Number of rows in df before modification:", len(df))

    # Step 1: Create a mapping of malicious node ID → most similar benign node ID

    malicious_to_benign = {}
    for i in nodeA:
        if labels[i] in top_nodes_by_similarity and len(top_nodes_by_similarity[labels[i]]) > 0:
            malicious_to_benign[mapp[i]] = mapp[top_nodes_by_similarity[labels[0]][0]]

    # Step 2: Retrieve all benign node IDs that will be replaced
    all_benign_replacements = set(malicious_to_benign.values())

    # Step 3: Extract relevant rows for each benign ID
    benign_rows_dict = {benign_id: df[(df['actorID'] == benign_id) | (df['objectID'] == benign_id)].copy()
                        for benign_id in all_benign_replacements}

    new_rows = []

    for malicious_id, benign_id in tqdm(malicious_to_benign.items(), desc="Processing malicious nodes"):
        if benign_id in benign_rows_dict:
            modified_rows = benign_rows_dict[benign_id].copy()  # Copy rows where benign ID appears
            modified_rows.loc[modified_rows['actorID'] == benign_id, 'actorID'] = malicious_id
            modified_rows.loc[modified_rows['objectID'] == benign_id, 'objectID'] = malicious_id
            new_rows.append(modified_rows)

    df_curated = pd.concat([df] + new_rows, ignore_index=True)

    print("Number of rows in df after modification:", len(df_curated))
    print("The number of triggered events are: ", len(df_curated) - len(df))

    return df_curated

df_curated = graph_modification(df, indices_of_malicious_nodes, top_nodes_by_similarity, mapp, labels)


In [ ]:
# Save the curated DataFrame to a text file (tab-separated, no header, no index)
output_path = "cadets_resources/cadets_test_curated.txt"
df_curated.to_csv(output_path, sep='\t', index=False, header=False)

print(f"Curated file saved as {output_path}")


## Evaluation after evasion

In [ ]:
# Load and process test dataset
path = 'cadets_resources/cadets_test_curated.txt' 
data_curated, feature_num, label_num, adj, adj2, nodeA = MyDatasetA(path, 'cadets')
dataset_curated = TestDatasetA(data_curated)
data_curated = dataset_curated[0] 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Load the whole model (architecture + weights)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

load_path = "../models/cadets/cadets_model_weights.pth"
model_good = SAGENet(feature_num, label_num).to(device)
model_good.load_state_dict(torch.load(load_path, map_location=device))

# Put in eval mode for inference
model_good.eval()
thre = 1.5

print("Model loaded successfully:", type(model_good))

acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask_evasion = test_one_model(data_curated.test_mask, model_good, device, data_curated, thre, nodeA)

print(f"Test Accuracy: {acc:.4f}")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, FPR: {fpr:.4f}")